<a href="https://colab.research.google.com/github/Ramdharshan2007/DAA-Lab-Experiment/blob/main/3C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import heapq
import time
import random

class GraphComparison:
    def __init__(self, vertices):
        self.V = vertices
        # Initialize adjacency matrix with 0s for O(V^2) approach
        self.matrix = [[0 for _ in range(vertices)] for _ in range(vertices)]
        # Initialize adjacency list for O((V+E) log V) heap approach
        self.adj_list = {i: [] for i in range(vertices)}

    def add_edge(self, u, v, weight):
        # Populate Adjacency Matrix
        self.matrix[u][v] = weight
        self.matrix[v][u] = weight
        # Populate Adjacency List
        self.adj_list[u].append((v, weight))
        self.adj_list[v].append((u, weight))

    # --- 1. Prim's Algorithm using Adjacency Matrix: O(V^2) ---
    def min_key(self, key, mst_set):
        min_val = sys.maxsize
        min_index = -1
        for v in range(self.V):
            if key[v] < min_val and not mst_set[v]:
                min_val = key[v]
                min_index = v
        return min_index

    def prim_matrix(self):
        key = [sys.maxsize] * self.V
        parent = [None] * self.V
        key[0] = 0
        mst_set = [False] * self.V
        parent[0] = -1

        for _ in range(self.V):
            # Pick the minimum key vertex not yet included in MST: O(V)
            u = self.min_key(key, mst_set)
            mst_set[u] = True

            # Update key and parent for adjacent vertices: O(V)
            for v in range(self.V):
                if self.matrix[u][v] > 0 and not mst_set[v] and key[v] > self.matrix[u][v]:
                    key[v] = self.matrix[u][v]
                    parent[v] = u

        # Calculate total cost
        total_cost = sum(self.matrix[i][parent[i]] for i in range(1, self.V))
        return total_cost

    # --- 2. Prim's Algorithm using Min-Heap and Adjacency List: O((V + E) log V) ---
    def prim_heap(self):
        key = [sys.maxsize] * self.V
        parent = [-1] * self.V
        mst_set = [False] * self.V
        key[0] = 0

        # Priority Queue stores tuples of (weight, vertex)
        pq = [(0, 0)]
        total_cost = 0

        while pq:
            w, u = heapq.heappop(pq) # O(log V)

            if mst_set[u]:
                continue
            mst_set[u] = True

            if parent[u] != -1:
                total_cost += w

            # Update adjacent vertices
            for v, weight in self.adj_list[u]:
                if not mst_set[v] and weight < key[v]:
                    key[v] = weight
                    parent[v] = u
                    heapq.heappush(pq, (key[v], v)) # O(log V)

        return total_cost

# --- Main Execution and Benchmarking ---
if __name__ == '__main__':
    V = 100
    g = GraphComparison(V)

    # Generate a Dense Graph (Edge between almost every pair of vertices)
    # Probability of an edge = 0.95
    edge_count = 0
    for i in range(V):
        for j in range(i + 1, V):
            if random.random() < 0.95:  # 95% chance to have an edge -> Highly Dense Graph
                weight = random.randint(1, 100)
                g.add_edge(i, j, weight)
                edge_count += 1

    print(f"Generated Dense Graph with {V} vertices and {edge_count} edges.\n")

    # Benchmark Matrix version
    start_time = time.perf_counter()
    cost_matrix = g.prim_matrix()
    matrix_time = time.perf_counter() - start_time

    # Benchmark Heap version
    start_time = time.perf_counter()
    cost_heap = g.prim_heap()
    heap_time = time.perf_counter() - start_time

    print(f"{'Algorithm Approach':<30} | {'MST Cost':<10} | {'Execution Time (s)'}")
    print("-" * 65)
    print(f"{'Prim (Adjacency Matrix)':<30} | {cost_matrix:<10} | {matrix_time:.6f}")
    print(f"{'Prim (Min-Heap + Adj List)':<30} | {cost_heap:<10} | {heap_time:.6f}")

    print("\n--- Theoretical Inference ---")
    print("For a dense graph where E approaches V^2:")
    print("- Adjacency Matrix time complexity: O(V^2)")
    print("- Min-Heap + Adj List time complexity: O((V + E) log V) ≈ O(V^2 log V)")
    print("Conclusion: For highly dense graphs, the simpler Adjacency Matrix approach")
    print("often outperforms the Heap-based approach because it avoids the O(log V)")
    print("overhead of priority queue operations for every edge evaluated.")

Generated Dense Graph with 100 vertices and 4702 edges.

Algorithm Approach             | MST Cost   | Execution Time (s)
-----------------------------------------------------------------
Prim (Adjacency Matrix)        | 175        | 0.001722
Prim (Min-Heap + Adj List)     | 175        | 0.001342

--- Theoretical Inference ---
For a dense graph where E approaches V^2:
- Adjacency Matrix time complexity: O(V^2)
- Min-Heap + Adj List time complexity: O((V + E) log V) ≈ O(V^2 log V)
Conclusion: For highly dense graphs, the simpler Adjacency Matrix approach
often outperforms the Heap-based approach because it avoids the O(log V)
overhead of priority queue operations for every edge evaluated.
